# 11 Hugging Face and Autorouting

This notebook audits raw-input auto-routing for Hugging Face text models. `tl.trace(hf_model, text)` can route through the input registry, and `tl.bridge.hf.trace_text` exposes the same path explicitly when a tokenizer is available.

The Hugging Face section uses `hf-internal-testing/tiny-random-distilbert`, a tiny fixture model. Every optional import and download step is guarded so the notebook executes cleanly when `transformers` or network access is unavailable.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import os
from typing import Any

import torch
from torch import nn
import torchlens as tl

os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "5")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "20")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
torch.manual_seed(47)

MODEL_ID = "hf-internal-testing/tiny-random-distilbert"
TEXT = "TorchLens traces tiny text models."

First inspect the built-in input detector registry. Lower priorities run first.

In [ ]:
for detector in tl.autoroute.input.list():
    print(f"{detector.priority:3d} {detector.name:14s} {detector.func.__name__}")

`tl.trace(model, text)` uses those detectors before falling back to ordinary tensor-style inputs. The cell below either runs a real tiny HF trace or records a clear skip note.

In [ ]:
hf_ready = False
hf_skip_reason = ""
hf_model: Any | None = None
hf_tokenizer: Any | None = None
hf_auto_trace: Any | None = None

try:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    hf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).eval()
    hf_auto_trace = tl.trace(hf_model, TEXT, layers_to_save="all")
    hf_ready = True
except Exception as exc:
    hf_skip_reason = f"{type(exc).__name__}: {exc}"
    print(
        "> NOTE: Hugging Face text tracing is skipped in this environment. "
        "Install transformers and allow the tiny model/tokenizer to resolve to run this section."
    )
    print(f"skip reason: {hf_skip_reason}")

if hf_ready:
    assert hf_auto_trace is not None
    print(f"model: {MODEL_ID}")
    print(f"trace type: {type(hf_auto_trace).__name__}")
    print(f"layers: {len(hf_auto_trace.layer_list)}")
    print(f"output layers: {hf_auto_trace.output_layers}")
    print(f"preprocessor: {hf_auto_trace.input_preprocessor.source}")

`tl.bridge.hf.trace_text(...)` is the explicit bridge helper. Passing the tokenizer avoids a second tokenizer resolution step.

In [ ]:
if not hf_ready:
    print(
        "> NOTE: bridge.hf.trace_text example skipped because the guarded HF setup did not complete."
    )
else:
    assert hf_model is not None
    assert hf_tokenizer is not None
    bridge_trace = tl.bridge.hf.trace_text(
        hf_model,
        TEXT,
        tokenizer=hf_tokenizer,
        layers_to_save="all",
    )
    print(f"bridge trace layers: {len(bridge_trace.layer_list)}")
    print(f"raw input: {bridge_trace.raw_input!r}")
    print(f"preprocessor id: {bridge_trace.input_preprocessor.identifier}")

Custom detectors can be registered on `tl.autoroute.input`. A detector returns `None` when it does not match, or returns a `Trace` when it claims the input.

In [ ]:
class AutorouteNet(nn.Module):
    """Tiny tensor model for a custom input detector."""

    def __init__(self) -> None:
        """Initialize the linear layer."""

        super().__init__()
        self.linear = nn.Linear(3, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the linear layer.

        Parameters
        ----------
        x:
            Tensor payload extracted by the custom detector.

        Returns
        -------
        torch.Tensor
            Linear output.
        """

        return self.linear(x)


with tl.autoroute.input.snapshot():

    @tl.autoroute.input.register(name="tensor_dict", priority=5)
    def tensor_dict_detector(model: nn.Module, payload: Any, **kwargs: Any) -> Any | None:
        """Route dictionaries containing a `tensor` value.

        Parameters
        ----------
        model:
            Model passed to `tl.trace`.
        payload:
            Raw user input.
        **kwargs:
            Trace options forwarded by the dispatcher.

        Returns
        -------
        Any | None
            A Trace when the payload matches, otherwise `None`.
        """

        if not (isinstance(payload, dict) and isinstance(payload.get("tensor"), torch.Tensor)):
            return None
        return tl.trace(model, payload["tensor"], **kwargs)

    print("registered order:")
    for detector in tl.autoroute.input.list()[:4]:
        print(f"{detector.priority:3d} {detector.name}")

    custom_model = AutorouteNet().eval()
    custom_trace = tl.trace(custom_model, {"tensor": torch.randn(1, 3)}, layers_to_save="all")
    print(f"custom trace layers: {len(custom_trace.layer_list)}")
    print(f"custom input layers: {custom_trace.input_layers}")

print(f"registry restored: {[detector.name for detector in tl.autoroute.input.list()]}")

> NOTE: `tl.autoroute.output` is reserved for a future sprint in this build; this notebook audits the currently executable input registry.

## Guarded Image and Multimodal Autorouting
`tl.bridge.hf.trace_image` and `tl.bridge.hf.trace_multimodal` exist in this build. Image/multimodal detector listing is available through `tl.autoroute.input`; separate `is_image_model` / `is_multimodal_model` helpers are not exported. The examples below are guarded so missing optional dependencies or network access produce precise notes.

In [ ]:
print(f"trace_image exported: {hasattr(tl.bridge.hf, 'trace_image')}")
print(f"trace_multimodal exported: {hasattr(tl.bridge.hf, 'trace_multimodal')}")
print(f"image detector helpers exported: {hasattr(tl.bridge.hf, 'is_image_model')}")
print(f"multimodal detector helpers exported: {hasattr(tl.bridge.hf, 'is_multimodal_model')}")
print(
    f"autoroute image/multimodal detectors: {[detector.name for detector in tl.autoroute.input.list() if 'image' in detector.name or 'multimodal' in detector.name]}"
)

try:
    from PIL import Image
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    image_id = "hf-internal-testing/tiny-random-vit"
    image = Image.new("RGB", (16, 16), color=(128, 64, 32))
    image_processor = AutoImageProcessor.from_pretrained(image_id)
    image_model = AutoModelForImageClassification.from_pretrained(image_id).eval()
    image_trace = tl.bridge.hf.trace_image(
        image_model,
        image,
        processor=image_processor,
        layers_to_save="all",
    )
    print(
        f"image trace layers: {len(image_trace.layers)} preprocessor={image_trace.input_preprocessor.identifier}"
    )
except Exception as exc:
    print(
        "> NOTE: guarded image tracing skipped; install pillow/transformers and allow the tiny "
        "image fixture to resolve to run this section."
    )
    print(f"skip reason: {type(exc).__name__}: {exc}")

try:
    from PIL import Image
    from transformers import AutoProcessor, AutoModelForVision2Seq

    mm_id = "hf-internal-testing/tiny-random-BlipForConditionalGeneration"
    processor = AutoProcessor.from_pretrained(mm_id)
    mm_model = AutoModelForVision2Seq.from_pretrained(mm_id).eval()
    mm_payload = {"text": "describe", "images": Image.new("RGB", (16, 16), color="white")}
    mm_trace = tl.bridge.hf.trace_multimodal(
        mm_model,
        mm_payload,
        processor=processor,
        layers_to_save="all",
    )
    print(
        f"multimodal trace layers: {len(mm_trace.layers)} preprocessor={mm_trace.input_preprocessor.identifier}"
    )
except Exception as exc:
    print(
        "> NOTE: guarded multimodal tracing skipped; install compatible transformers deps and "
        "allow the tiny multimodal fixture to resolve to run this section."
    )
    print(f"skip reason: {type(exc).__name__}: {exc}")

## 🔧 Sandbox

Mini-experiment: change `sandbox_key`, payload tensor values, and detector priority. Expected observation: inside the snapshot the custom detector routes the dict payload; after the cell, registry state is restored.

In [ ]:
sandbox_priority = 8
sandbox_key = "tensor"
with tl.autoroute.input.snapshot():

    @tl.autoroute.input.register(name="sandbox_tensor_dict", priority=sandbox_priority)
    def sandbox_detector(model: nn.Module, payload: Any, **kwargs: Any) -> Any | None:
        """Route a sandbox tensor dictionary.

        Parameters
        ----------
        model:
            Model passed to `tl.trace`.
        payload:
            Raw user input.
        **kwargs:
            Trace options forwarded by the dispatcher.

        Returns
        -------
        Any | None
            A Trace when the payload contains the sandbox key.
        """

        if not (isinstance(payload, dict) and isinstance(payload.get(sandbox_key), torch.Tensor)):
            return None
        return tl.trace(model, payload[sandbox_key], **kwargs)

    sandbox_model = AutorouteNet().eval()
    sandbox_log = tl.trace(
        sandbox_model, {sandbox_key: torch.full((1, 3), 2.0)}, layers_to_save="all"
    )
    print(f"sandbox detectors: {[detector.name for detector in tl.autoroute.input.list()[:3]]}")
    print(f"sandbox output shape: {sandbox_log[sandbox_log.output_layers[0]].shape}")
print(f"detector priority used: {sandbox_priority}")